# Colab: run a checked-in sEMG experiment

Select a hosted GPU and run the settings and launch cells. The baseline runs by default. Choose a different manifest through `EXPERIMENT`. A short smoke test is available through `SMOKE_TEST`. See the repository README for the data split, artifact paths, and collaboration workflow.


In [ ]:
print("Colab kernel is responding")

In [ ]:
REPO_URL = "https://github.com/EdwinKam/emg2qwerty.git"
PROJECT_REF = "HEAD"  # Bundle HEAD, or a specific published commit SHA.
EXPERIMENT = "benchmarks/tds_baseline/experiment.yaml"
SMOKE_TEST = False

DRIVE_PROJECT = "/content/drive/MyDrive/ECE247A_final_project"
DRIVE_ZIP = f"{DRIVE_PROJECT}/89335547.zip"
PROJECT_BUNDLE = f"{DRIVE_PROJECT}/emg2qwerty-experiments.bundle"  # Set to None for GitHub.
EXPERIMENT_OUTPUT_ROOT = f"{DRIVE_PROJECT}/experiments/manifest-runs"
RELEASE_GPU_ON_SUCCESS = True


In [ ]:
import os, subprocess, sys, time, threading, signal, zipfile, shutil, tempfile
from datetime import datetime
from pathlib import Path
try:
    import google.colab
    import torch
except ImportError as exc:
    raise RuntimeError('Select a hosted Colab GPU kernel first.') from exc
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before continuing.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
# Keep the subprocess environment compatible with Colab and print complete errors.
os.environ['MPLBACKEND'] = 'Agg'
os.environ['HYDRA_FULL_ERROR'] = '1'
root = Path('/content')
repo = root / 'emg2qwerty-experiments'
data = root / 'emg2qwerty-subject-89335547'
archive = root / 'subject-89335547.zip'
env = root / 'emg2qwerty-env'
mamba = root / 'bin/micromamba'

status_path = root / 'emg2qwerty-status.json'
def status(stage, state='running', detail=''):
    record = {'stage': stage, 'state': state, 'detail': detail, 'updated': datetime.now().astimezone().isoformat()}
    temporary = status_path.with_suffix('.tmp')
    temporary.write_text(__import__('json').dumps(record, indent=2))
    temporary.replace(status_path)
    print(f"[{record['updated']}] {stage}: {state} {detail}", flush=True)
def run(args, stage='Setup', cwd=None, env=None, log_path=None):
    status(stage, detail='Starting: ' + ' '.join(map(str, args)))
    started = time.monotonic()
    process = subprocess.Popen([str(x) for x in args], cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1,
                               start_new_session=True)
    done = threading.Event()
    def heartbeat():
        while not done.wait(30):
            print(f'[{stage}] Still running ({int(time.monotonic()-started)} seconds elapsed)...', flush=True)
    thread = threading.Thread(target=heartbeat, daemon=True)
    thread.start()
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            if log_path:
                with open(log_path, 'a') as log:
                    log.write(line)
        rc = process.wait()
        if rc:
            status(stage, 'failed', f'Exit code {rc}')
            raise subprocess.CalledProcessError(rc, args)
        status(stage, 'completed', f'{int(time.monotonic()-started)} seconds')
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        status(stage, 'interrupted', 'Subprocess stopped')
        raise
    finally:
        done.set()
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
project_source = Path(PROJECT_BUNDLE) if PROJECT_BUNDLE else REPO_URL
if PROJECT_BUNDLE and not project_source.is_file():
    raise FileNotFoundError(f'Put the project Git bundle at {project_source}.')
if not repo.exists():
    run(['git', 'clone', project_source, repo], env=git_env, stage='Clone repository')
if subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain', '--untracked-files=no'], text=True).strip():
    raise RuntimeError(f'{repo} has local changes; preserve them before continuing.')
run(['git', '-C', repo, 'fetch', project_source, PROJECT_REF], env=git_env, stage='Fetch selected commit')
run(['git', '-C', repo, 'checkout', '--detach', 'FETCH_HEAD'], env=git_env)
print('Code commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip(), flush=True)
runner = repo / 'scripts/run_experiment.py'
experiment = repo / EXPERIMENT
if not experiment.is_file() or not runner.is_file():
    raise FileNotFoundError(f'Check PROJECT_REF and EXPERIMENT. Missing runner or {experiment}.')
if not mamba.exists():
    run(['bash', '-c', 'set -o pipefail; curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba'])
if not (env / 'bin/python').exists():
    run([mamba, 'create', '-y', '-p', env, '-c', 'conda-forge', 'python=3.10', 'pip=24.0'], stage='Create Python environment')
python = env / 'bin/python'
marker = env / '.dependencies-ready'
if not marker.exists():
    run([python, '-m', 'pip', 'install', '-r', repo / 'requirements.txt'], stage='Install dependencies')
    run([python, '-c', 'import torch, torchaudio, pytorch_lightning, hydra, kenlm; assert torch.cuda.is_available(), "PyTorch cannot access GPU"'])
    marker.touch()
run([python, '-m', 'pip', 'install', '--no-deps', '-e', repo], stage='Install project checkout')
launch = [python, runner, '--experiment', experiment, '--data', data]
def data_ready():
    result = subprocess.run([*launch, '--check-data'], capture_output=True, text=True)
    print(result.stdout, end='', flush=True)
    print(result.stderr, end='', flush=True)
    if result.returncode not in (0, 1):
        raise RuntimeError(f'Dataset check failed with exit code {result.returncode}; see details above.')
    return result.returncode == 0
if not data_ready():
    if not archive.exists():
        source = Path(DRIVE_ZIP)
        if not source.is_file():
            raise FileNotFoundError(f'Subject ZIP not found: {source}. Update DRIVE_ZIP in the settings cell.')
        status('Copy subject ZIP from Drive', detail=f'{source} ({source.stat().st_size / 1e9:.2f} GB)')
        copied, total, last_report = 0, source.stat().st_size, 0
        with source.open('rb') as src, tempfile.NamedTemporaryFile(dir=root, prefix='subject-copy-', suffix='.part', delete=False) as dest:
            temporary_path = Path(dest.name)
            try:
                while True:
                    chunk = src.read(1024 * 1024)
                    if not chunk: break
                    dest.write(chunk)
                    copied += len(chunk)
                    if time.monotonic() - last_report >= 10:
                        status('Copy subject ZIP from Drive', detail=f'{copied / 1e9:.2f}/{total / 1e9:.2f} GB ({100 * copied / total:.1f}%)')
                        last_report = time.monotonic()
            except BaseException:
                print(f'Copy interrupted. Partial file kept at {temporary_path}', flush=True)
                raise
        if copied != total or not zipfile.is_zipfile(temporary_path):
            raise RuntimeError(f'The Drive copy is incomplete or invalid: {temporary_path}')
        temporary_path.replace(archive)
        status('Copy subject ZIP from Drive', 'completed', f'{copied / 1e9:.2f} GB')
    if not zipfile.is_zipfile(archive):
        raise RuntimeError(f'{archive} is not a valid ZIP; inspect the file before retrying.')
    # Copy only session files into the dedicated data directory, without overwriting existing files.
    data.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        members = [m for m in zf.infolist() if not m.is_dir() and m.filename.lower().endswith('.hdf5')]
        if not members:
            raise RuntimeError('The ZIP contains no .hdf5 session files; inspect its contents.')
        names = [Path(m.filename).name for m in members]
        if len(names) != len(set(names)):
            raise RuntimeError('The ZIP contains duplicate HDF5 filenames; inspect it before extracting.')
        for index, member in enumerate(members, 1):
            dest = data / Path(member.filename).name
            if dest.exists():
                print(f'Preserving existing file: {dest}', flush=True)
                continue
            temp = dest.with_name(dest.name + '.part')
            if temp.exists():
                raise RuntimeError(f'Partial extraction exists at {temp}; inspect it before retrying.')
            status('Extract subject data', detail=f'{index}/{len(members)}: {dest.name}')
            with zf.open(member) as source, temp.open('xb') as target:
                shutil.copyfileobj(source, target, length=1024 * 1024)
            temp.replace(dest)
    if not data_ready():
        raise RuntimeError('Subject ZIP is missing configured sessions. Check DRIVE_ZIP against the selected experiment split.')
status('Dataset verification', 'completed', 'Required train, validation and test files found')
command = [*launch, '--output-root', EXPERIMENT_OUTPUT_ROOT]
if SMOKE_TEST:
    command.append('--smoke')
run(command, cwd=repo, stage=f'Experiment {EXPERIMENT}')
status('Results', 'completed', f'Find the experiment report under {EXPERIMENT_OUTPUT_ROOT}.')
if RELEASE_GPU_ON_SUCCESS and not SMOKE_TEST:
    from google.colab import runtime
    drive.flush_and_unmount()
    runtime.unassign()


## Check status after stopping a cell

Run the next cell on the same server if the launch cell has ended. During a run, read the latest `summary.md` or case `console.log` in Drive.


In [ ]:
from pathlib import Path

status_file = Path("/content/emg2qwerty-status.json")
print(status_file.read_text() if status_file.is_file() else "No status for this runtime.")
reports = sorted(
    Path(EXPERIMENT_OUTPUT_ROOT).glob("*/summary.md"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if reports:
    print(reports[0])
    print(reports[0].read_text())


## Save results and stop the GPU

The runner saves reports and checkpoints under `EXPERIMENT_OUTPUT_ROOT` in Drive. A successful full run flushes Drive and releases the GPU by default. Smoke tests leave the runtime available for a full run. If you stop or a run fails, release the GPU once it is idle. Resume with the same commit, manifest, seed, and output path.
